In [1]:
"""
Global Superstore - Phase 2 | Notebook 01
Exploratory Data Analysis (EDA)
File: 02_Python/01_EDA.py

Chay: python 02_Python/01_EDA.py
Ket qua: cac bieu do luu trong thu muc 02_Python/charts/
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import sqlite3
import os
import warnings
warnings.filterwarnings("ignore")

# ── Setup ─────────────────────────────────────────────────────────────────────
DB_FILE    = r"C:\DA\global-superstore-analysis\data\superstore.db"
CHART_DIR  = r"C:\DA\global-superstore-analysis\02_Python\charts"
os.makedirs(CHART_DIR, exist_ok=True)

# Style
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "font.size":        11,
})
COLORS = ["#2E75B6", "#ED7D31", "#70AD47", "#FFC000", "#FF0000", "#9E480E"]

print("=" * 60)
print("GLOBAL SUPERSTORE - EDA")
print("=" * 60)

# ── Load data ──────────────────────────────────────────────────────────────────
conn = sqlite3.connect(DB_FILE)
df   = pd.read_sql("SELECT * FROM orders", conn)
conn.close()

df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"]  = pd.to_datetime(df["ship_date"])

print(f"\nData loaded: {df.shape[0]:,} rows x {df.shape[1]} cols")
print(f"Time range : {df['order_date'].min().date()} -> {df['order_date'].max().date()}")
print(f"Countries  : {df['country'].nunique()}")
print(f"Customers  : {df['customer_id'].nunique():,}")
print(f"Products   : {df['product_id'].nunique():,}")

# ── CHART 1: Revenue & Profit by Category ────────────────────────────────────
print("\n[Chart 1] Revenue & Profit by Category...")

cat_df = df.groupby("category").agg(
    total_sales  = ("sales",  "sum"),
    total_profit = ("profit", "sum")
).reset_index()
cat_df["margin_pct"] = cat_df["total_profit"] / cat_df["total_sales"] * 100
cat_df = cat_df.sort_values("total_sales", ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(cat_df["category"], cat_df["total_sales"]/1e6,
               color=COLORS[:3], height=0.5)
ax2 = ax.twiny()
ax2.barh(cat_df["category"], cat_df["total_profit"]/1e6,
         color=[c + "88" for c in COLORS[:3]], height=0.3)

for bar, row in zip(bars, cat_df.itertuples()):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f"${row.total_sales/1e6:.1f}M  |  Margin: {row.margin_pct:.1f}%",
            va="center", fontsize=10)

ax.set_xlabel("Total Sales (Million USD)")
ax.set_title("Chart 1 — Revenue & Profit Margin by Category", fontsize=13, fontweight="bold", pad=12)
ax.legend(["Revenue"], loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "chart_01_category.png"), dpi=150)
plt.close()
print("   Saved: chart_01_category.png")

# ── CHART 2: Monthly Revenue Trend (2011-2014) ────────────────────────────────
print("[Chart 2] Monthly Revenue Trend...")

df["year_month"] = df["order_date"].dt.to_period("M")
monthly = df.groupby("year_month").agg(
    sales  = ("sales",  "sum"),
    profit = ("profit", "sum")
).reset_index()
monthly["year_month_dt"] = monthly["year_month"].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(monthly["year_month_dt"], monthly["sales"]/1e3,
                alpha=0.15, color=COLORS[0])
ax.plot(monthly["year_month_dt"], monthly["sales"]/1e3,
        color=COLORS[0], linewidth=2, label="Revenue")
ax2 = ax.twinx()
ax2.plot(monthly["year_month_dt"], monthly["profit"]/1e3,
         color=COLORS[1], linewidth=1.5, linestyle="--", label="Profit")

# Danh dau Q4 peaks
for year in [2011, 2012, 2013, 2014]:
    q4 = monthly[monthly["year_month_dt"].dt.year == year]
    if not q4.empty:
        peak = q4.loc[q4["sales"].idxmax()]
        ax.annotate(f"Q4/{year}",
                    xy=(peak["year_month_dt"], peak["sales"]/1e3),
                    xytext=(0, 12), textcoords="offset points",
                    ha="center", fontsize=8, color=COLORS[0],
                    arrowprops=dict(arrowstyle="-", color="gray", lw=0.8))

ax.set_ylabel("Revenue (K USD)", color=COLORS[0])
ax2.set_ylabel("Profit (K USD)", color=COLORS[1])
ax.set_title("Chart 2 — Monthly Revenue & Profit Trend (2011–2014)",
             fontsize=13, fontweight="bold", pad=12)
lines1, l1 = ax.get_legend_handles_labels()
lines2, l2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, l1 + l2, loc="upper left")
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "chart_02_monthly_trend.png"), dpi=150)
plt.close()
print("   Saved: chart_02_monthly_trend.png")

# ── CHART 3: Profit Margin by Sub-Category ────────────────────────────────────
print("[Chart 3] Sub-Category Profit Margin...")

sub_df = df.groupby(["category", "sub_category"]).agg(
    sales  = ("sales",  "sum"),
    profit = ("profit", "sum")
).reset_index()
sub_df["margin"] = sub_df["profit"] / sub_df["sales"] * 100
sub_df = sub_df.sort_values("margin")

colors_bar = ["#E74C3C" if m < 0 else "#2E75B6" for m in sub_df["margin"]]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(sub_df["sub_category"], sub_df["margin"], color=colors_bar, height=0.6)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")

for bar, val in zip(bars, sub_df["margin"]):
    x = bar.get_width()
    ax.text(x + (0.3 if x >= 0 else -0.3), bar.get_y() + bar.get_height()/2,
            f"{val:.1f}%", va="center", ha="left" if x >= 0 else "right",
            fontsize=9)

ax.set_xlabel("Profit Margin (%)")
ax.set_title("Chart 3 — Profit Margin by Sub-Category\n(Red = Loss-making)",
             fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "chart_03_subcategory_margin.png"), dpi=150)
plt.close()
print("   Saved: chart_03_subcategory_margin.png")

# ── CHART 4: Discount vs Profit Scatter ───────────────────────────────────────
print("[Chart 4] Discount vs Profit...")

sample = df.sample(min(5000, len(df)), random_state=42)
colors_scatter = [COLORS[0] if p >= 0 else COLORS[4] for p in sample["profit"]]

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(sample["discount"] * 100, sample["profit"],
           c=colors_scatter, alpha=0.35, s=15)
ax.axhline(0,   color="red",   linewidth=1.2, linestyle="--", label="Break-even")
ax.axvline(20,  color="orange", linewidth=1.2, linestyle=":", label="20% discount")
ax.axvline(40,  color="red",    linewidth=1.2, linestyle=":", label="40% discount")

# Them trend line
z = np.polyfit(sample["discount"]*100, sample["profit"], 1)
p = np.poly1d(z)
x_line = np.linspace(0, 90, 100)
ax.plot(x_line, p(x_line), color="navy", linewidth=2, label="Trend")

ax.set_xlabel("Discount (%)")
ax.set_ylabel("Profit (USD)")
ax.set_title("Chart 4 — Discount vs Profit\n(Blue=Profit, Red=Loss)",
             fontsize=13, fontweight="bold", pad=12)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "chart_04_discount_profit.png"), dpi=150)
plt.close()
print("   Saved: chart_04_discount_profit.png")

# ── CHART 5: Revenue by Region ────────────────────────────────────────────────
print("[Chart 5] Revenue by Region...")

region_df = df.groupby("region").agg(
    sales  = ("sales",  "sum"),
    profit = ("profit", "sum"),
    orders = ("order_id", "nunique")
).reset_index()
region_df["margin"] = region_df["profit"] / region_df["sales"] * 100
region_df = region_df.sort_values("sales", ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

bars = ax1.bar(range(len(region_df)), region_df["sales"]/1e6,
               color=COLORS[:len(region_df)], width=0.6)
ax1.set_xticks(range(len(region_df)))
ax1.set_xticklabels(region_df["region"], rotation=30, ha="right", fontsize=9)
ax1.set_ylabel("Revenue (Million USD)")
ax1.set_title("Revenue by Region", fontweight="bold")
for bar, val in zip(bars, region_df["sales"]):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"${val/1e6:.1f}M", ha="center", fontsize=8)

colors_margin = ["#E74C3C" if m < 0 else "#70AD47" for m in region_df["margin"]]
ax2.bar(range(len(region_df)), region_df["margin"],
        color=colors_margin, width=0.6)
ax2.axhline(0, color="black", linewidth=0.8, linestyle="--")
for i, val in enumerate(region_df["margin"]):
    ax2.text(i, val + (0.5 if val >= 0 else -0.5), f"{val:.1f}%",
             ha="center", va="bottom" if val >= 0 else "top", fontsize=8)
ax2.set_xticks(range(len(region_df)))
ax2.set_xticklabels(region_df["region"], rotation=30, ha="right", fontsize=9)
ax2.set_ylabel("Profit Margin (%)")
ax2.set_title("Profit Margin by Region", fontweight="bold")

plt.suptitle("Chart 5 — Revenue & Profit Margin by Region",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "chart_05_region.png"),
            dpi=150, bbox_inches="tight")
plt.close()
print("   Saved: chart_05_region.png")

# ── CHART 6: Yearly Growth ────────────────────────────────────────────────────
print("[Chart 6] Yearly Revenue Growth...")

yearly = df.groupby("year").agg(
    sales   = ("sales",  "sum"),
    profit  = ("profit", "sum"),
    orders  = ("order_id", "nunique"),
    customers = ("customer_id", "nunique")
).reset_index()
yearly["yoy_growth"] = yearly["sales"].pct_change() * 100

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(yearly["year"].astype(str), yearly["sales"]/1e6,
              color=COLORS[:4], width=0.5)
ax2 = ax.twinx()
ax2.plot(yearly["year"].astype(str), yearly["yoy_growth"],
         color="red", marker="o", linewidth=2, label="YoY Growth %")
ax2.axhline(0, color="gray", linewidth=0.5)

for bar, row in zip(bars, yearly.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"${row.sales/1e6:.1f}M", ha="center", fontsize=10, fontweight="bold")

ax.set_ylabel("Revenue (Million USD)")
ax2.set_ylabel("YoY Growth (%)", color="red")
ax.set_title("Chart 6 — Yearly Revenue & YoY Growth",
             fontsize=13, fontweight="bold", pad=12)
ax2.legend(loc="upper left")
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "chart_06_yearly_growth.png"), dpi=150)
plt.close()
print("   Saved: chart_06_yearly_growth.png")

# ── SUMMARY ────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("EDA SUMMARY — KEY INSIGHTS")
print("=" * 60)

total_sales  = df["sales"].sum()
total_profit = df["profit"].sum()
margin       = total_profit / total_sales * 100

print(f"\n Total Revenue  : ${total_sales/1e6:.2f}M")
print(f" Total Profit   : ${total_profit/1e6:.2f}M")
print(f" Overall Margin : {margin:.1f}%")

# Top profitable sub-category
top_sub = sub_df.nlargest(1, "margin")
loss_sub = sub_df[sub_df["margin"] < 0]
print(f"\n Best sub-category : {top_sub.iloc[0]['sub_category']} ({top_sub.iloc[0]['margin']:.1f}% margin)")
print(f" Loss-making subs  : {len(loss_sub)} sub-categories")
for _, row in loss_sub.iterrows():
    print(f"   -> {row['sub_category']} ({row['margin']:.1f}%)")

# Discount insight
high_disc = df[df["discount"] >= 0.4]
print(f"\n Orders with discount >= 40% : {len(high_disc):,}")
print(f" Avg profit when disc >= 40%  : ${high_disc['profit'].mean():.1f}")
print(f" % of orders that are loss    : {(df['profit'] < 0).mean()*100:.1f}%")

print(f"\n Charts saved to: {CHART_DIR}")
print("\n" + "=" * 60)
print("HOAN THANH Phase 2 - Notebook 01!")
print("Buoc tiep theo: 02_Python/02_Discount_Analysis.py")
print("=" * 60)

GLOBAL SUPERSTORE - EDA

Data loaded: 51,290 rows x 28 cols
Time range : 2011-01-01 -> 2014-12-31
Countries  : 147
Customers  : 1,590
Products   : 10,292

[Chart 1] Revenue & Profit by Category...
   Saved: chart_01_category.png
[Chart 2] Monthly Revenue Trend...
   Saved: chart_02_monthly_trend.png
[Chart 3] Sub-Category Profit Margin...
   Saved: chart_03_subcategory_margin.png
[Chart 4] Discount vs Profit...
   Saved: chart_04_discount_profit.png
[Chart 5] Revenue by Region...
   Saved: chart_05_region.png
[Chart 6] Yearly Revenue Growth...
   Saved: chart_06_yearly_growth.png

EDA SUMMARY — KEY INSIGHTS

 Total Revenue  : $12.64M
 Total Profit   : $1.47M
 Overall Margin : 11.6%

 Best sub-category : Paper (24.2% margin)
 Loss-making subs  : 1 sub-categories
   -> Tables (-8.5%)

 Orders with discount >= 40% : 10,138
 Avg profit when disc >= 40%  : $-76.1
 % of orders that are loss    : 24.5%

 Charts saved to: C:\DA\global-superstore-analysis\02_Python\charts

HOAN THANH Phase 2 - 